Exploring the Data collected from the ONet

In [3]:
import pandas as pd

# Load O*NET datasets
occupation = pd.read_csv(
    r"C:\Users\belie\OneDrive\Desktop\ADT MySQL\Lab 7 Project Milestone\JobAvenir\data\Onet\occupation_data.csv"
)

software_skills = pd.read_csv(
    r"C:\Users\belie\OneDrive\Desktop\ADT MySQL\Lab 7 Project Milestone\JobAvenir\data\Onet\software_skills.csv"
)

job_titles = pd.read_csv(
    r"C:\Users\belie\OneDrive\Desktop\ADT MySQL\Lab 7 Project Milestone\JobAvenir\data\Onet\job_titles.csv"
)

# Display dataset information
datasets = {
    "Occupation Data": occupation,
    "Software Skills": software_skills,
    "Job Titles": job_titles
}

for name, df in datasets.items():

    print("\nDataset:", name)

    print("Rows and Columns:", df.shape)

    print("Column Names:")
    print(df.columns.tolist())

    print("Missing Values:")
    print(df.isnull().sum())

    print("Duplicate Rows:", df.duplicated().sum())

    display(df.head())


Dataset: Occupation Data
Rows and Columns: (1016, 3)
Column Names:
['O*NET-SOC Code', 'Title', 'Description']
Missing Values:
O*NET-SOC Code    0
Title             0
Description       0
dtype: int64
Duplicate Rows: 0


,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."



Dataset: Software Skills
Rows and Columns: (31821, 7)
Column Names:
['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element ID', 'Element Name', 'Hot Technology', 'In Demand']
Missing Values:
O*NET-SOC Code       0
Title                0
Workplace Example    0
Element ID           0
Element Name         0
Hot Technology       0
In Demand            0
dtype: int64
Duplicate Rows: 0


,O*NET-SOC Code,Title,Workplace Example,Element ID,Element Name,Hot Technology,In Demand
0,11-1011.00,Chief Executives,Adobe Acrobat,2.E.5.b,Document management software,Y,N
1,11-1011.00,Chief Executives,AdSense Tracker,2.E.6.f,Data base user interface and query software,N,N
2,11-1011.00,Chief Executives,Atlassian JIRA,2.E.5.a,Content workflow software,Y,N
3,11-1011.00,Chief Executives,Blackbaud The Raiser's Edge,2.E.6.c,Customer relationship management CRM software,N,N
4,11-1011.00,Chief Executives,ComputerEase construction accounting software,2.E.2.a,Accounting software,N,N



Dataset: Job Titles
Rows and Columns: (54269, 5)
Column Names:
['O*NET-SOC Code', 'Title', 'Job Title', 'Short Title', 'Source(s)']
Missing Values:
O*NET-SOC Code        0
Title                 0
Job Title             0
Short Title       49814
Source(s)             0
dtype: int64
Duplicate Rows: 0


,O*NET-SOC Code,Title,Job Title,Short Title,Source(s)
0,11-1011.00,Chief Executives,Aeronautics Commission Director,NaN,08
1,11-1011.00,Chief Executives,Agency Owner,NaN,10
2,11-1011.00,Chief Executives,Agricultural Services Director,NaN,08
3,11-1011.00,Chief Executives,Arts and Humanities Council Director,NaN,08
4,11-1011.00,Chief Executives,Bank President,NaN,09


Step 1:-

What are we going to clean?

Cleaning Task	                          Why It's Needed

Rename columns	                          Make column names suitable for Snowflake SQL
Remove unnecessary spaces	              Improve matching between datasets
Handle missing values	                  Prevent problems when loading data
Check duplicate records	                  Avoid storing repeated data
Standardize job titles	                  Compare similar technology roles
Clean HTML descriptions	                  Make job descriptions readable
Standardize skill names	                  Match skills across Jobicy and O*NET
Validate occupation codes	              Ensure relationships between tables are correct

In [2]:
import pandas as pd

# Folder containing your O*NET datasets
folder = r"C:\Users\belie\OneDrive\Desktop\ADT MySQL\Lab 7 Project Milestone\JobAvenir\data\Onet"

# Load Occupation Data
occupation = pd.read_csv(
    folder + r"\occupation_data.csv"
)

# Load Software Skills Data
software_skills = pd.read_csv(
    folder + r"\software_skills.csv"
)

# Load Job Titles Data
job_titles = pd.read_csv(
    folder + r"\job_titles.csv",
    dtype={"Source(s)": "string"}
)

# Verify that all datasets are loaded
print("Occupation Data:", occupation.shape)
print("Software Skills:", software_skills.shape)
print("Job Titles:", job_titles.shape)

Occupation Data: (1016, 3)
Software Skills: (31821, 7)
Job Titles: (54269, 5)


In [3]:
import pandas as pd

# Make a copy of the original dataset
occupation_clean = occupation.copy()

# Rename columns for Snowflake
occupation_clean = occupation_clean.rename(columns={
    "O*NET-SOC Code": "occupation_id",
    "Title": "occupation_title",
    "Description": "description"
})

# Remove unnecessary spaces
occupation_clean["occupation_id"] = (
    occupation_clean["occupation_id"]
    .astype("string")
    .str.strip()
)

occupation_clean["occupation_title"] = (
    occupation_clean["occupation_title"]
    .astype("string")
    .str.strip()
)

occupation_clean["description"] = (
    occupation_clean["description"]
    .astype("string")
    .str.strip()
)

# Remove completely identical rows
occupation_clean = occupation_clean.drop_duplicates()

# Check for duplicate primary keys
duplicate_ids = occupation_clean[
    occupation_clean["occupation_id"].duplicated(keep=False)
]

print("Cleaned Occupation Records:", len(occupation_clean))
print("Duplicate Occupation IDs:", len(duplicate_ids))

display(occupation_clean.head())

Cleaned Occupation Records: 1016
Duplicate Occupation IDs: 0


,occupation_id,occupation_title,description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."


What have we done?

Original Column                Cleaned Column

O*NET-SOC Code                 occupation_id 
Title                          occupation_title 
Description                    description

We have also checked whether any occupation identifiers are duplicated.

We will use occupation_id as the primary key for the OCCUPATIONS table.

Step 3: Clean Software Skills Data

In [4]:
# Make a copy
software_clean = software_skills.copy()

# Rename columns
software_clean = software_clean.rename(columns={
    "O*NET-SOC Code": "occupation_id",
    "Title": "occupation_title",
    "Workplace Example": "skill_name",
    "Element ID": "element_id",
    "Element Name": "skill_category",
    "Hot Technology": "hot_technology",
    "In Demand": "in_demand"
})

# Remove unnecessary spaces from text columns
for column in software_clean.select_dtypes(
    include=["object", "string"]
).columns:

    software_clean[column] = (
        software_clean[column]
        .astype("string")
        .str.strip()
    )

# Standardize Y/N values
software_clean["hot_technology"] = (
    software_clean["hot_technology"].str.upper()
)

software_clean["in_demand"] = (
    software_clean["in_demand"].str.upper()
)

# Remove completely identical rows
software_clean = software_clean.drop_duplicates()

print("Cleaned Software Skills Records:", len(software_clean))

display(software_clean.head())

Cleaned Software Skills Records: 31821


,occupation_id,occupation_title,skill_name,element_id,skill_category,hot_technology,in_demand
0,11-1011.00,Chief Executives,Adobe Acrobat,2.E.5.b,Document management software,Y,N
1,11-1011.00,Chief Executives,AdSense Tracker,2.E.6.f,Data base user interface and query software,N,N
2,11-1011.00,Chief Executives,Atlassian JIRA,2.E.5.a,Content workflow software,Y,N
3,11-1011.00,Chief Executives,Blackbaud The Raiser's Edge,2.E.6.c,Customer relationship management CRM software,N,N
4,11-1011.00,Chief Executives,ComputerEase construction accounting software,2.E.2.a,Accounting software,N,N


Step 4: Clean Job Titles Data

In [5]:
# Make a copy
job_titles_clean = job_titles.copy()

# Rename columns
job_titles_clean = job_titles_clean.rename(columns={
    "O*NET-SOC Code": "occupation_id",
    "Title": "occupation_title",
    "Job Title": "job_title",
    "Short Title": "short_title",
    "Source(s)": "source"
})

# Remove unnecessary spaces
for column in [
    "occupation_id",
    "occupation_title",
    "job_title",
    "short_title"
]:

    job_titles_clean[column] = (
        job_titles_clean[column]
        .astype("string")
        .str.strip()
    )

# Replace empty strings with missing values
job_titles_clean["short_title"] = (
    job_titles_clean["short_title"]
    .replace("", pd.NA)
)

# Remove completely identical rows
job_titles_clean = job_titles_clean.drop_duplicates()

print("Cleaned Job Titles Records:", len(job_titles_clean))

print(
    "Missing Short Titles:",
    job_titles_clean["short_title"].isna().sum()
)

display(job_titles_clean.head())

Cleaned Job Titles Records: 54269
Missing Short Titles: 49814


,occupation_id,occupation_title,job_title,short_title,source
0,11-1011.00,Chief Executives,Aeronautics Commission Director,<NA>,08
1,11-1011.00,Chief Executives,Agency Owner,<NA>,10
2,11-1011.00,Chief Executives,Agricultural Services Director,<NA>,08
3,11-1011.00,Chief Executives,Arts and Humanities Council Director,<NA>,08
4,11-1011.00,Chief Executives,Bank President,<NA>,09


Step 5: Validate connections between the three O*NET datasets

In [6]:
# Get valid occupation IDs
valid_occupation_ids = set(
    occupation_clean["occupation_id"].dropna()
)

# Check Software Skills
invalid_software = software_clean[
    ~software_clean["occupation_id"].isin(
        valid_occupation_ids
    )
]

# Check Job Titles
invalid_titles = job_titles_clean[
    ~job_titles_clean["occupation_id"].isin(
        valid_occupation_ids
    )
]

print("Invalid Software Skills Records:", len(invalid_software))

print("Invalid Job Titles Records:", len(invalid_titles))

Invalid Software Skills Records: 0
Invalid Job Titles Records: 0


In [ ]:
Step 6: Connect O*NET Occupations with Software Skills

In [7]:
# Connect Occupations with Software Skills
occupation_skills = pd.merge(
    occupation_clean[
        ["occupation_id", "occupation_title"]
    ],

    software_clean[
        [
            "occupation_id",
            "skill_name",
            "hot_technology",
            "in_demand"
        ]
    ],

    on="occupation_id",

    how="inner",

    validate="one_to_many"
)

print("Total Occupation-Skill Records:", len(occupation_skills))

display(occupation_skills.head(10))

Total Occupation-Skill Records: 31821


,occupation_id,occupation_title,skill_name,hot_technology,in_demand
0,11-1011.00,Chief Executives,Adobe Acrobat,Y,N
1,11-1011.00,Chief Executives,AdSense Tracker,N,N
2,11-1011.00,Chief Executives,Atlassian JIRA,Y,N
3,11-1011.00,Chief Executives,Blackbaud The Raiser's Edge,N,N
4,11-1011.00,Chief Executives,ComputerEase construction accounting software,N,N
5,11-1011.00,Chief Executives,Database reporting software,N,N
6,11-1011.00,Chief Executives,Databox,N,N
7,11-1011.00,Chief Executives,Email software,N,N
8,11-1011.00,Chief Executives,Exact Software Macola ES Labor Performance,N,N
9,11-1011.00,Chief Executives,Extensible markup language XML,Y,N


What does this do?

It connects occupation information with associated software technologies.

For example, you can find all the technologies linked to Data Scientists.

In [8]:
data_scientist_skills = occupation_skills[
    occupation_skills["occupation_title"].str.contains(
        "Data Scientists",
        case=False,
        na=False
    )
]

display(
    data_scientist_skills[
        [
            "occupation_title",
            "skill_name",
            "hot_technology",
            "in_demand"
        ]
    ]
)

,occupation_title,skill_name,hot_technology,in_demand
11865,Data Scientists,Alteryx software,Y,N
11866,Data Scientists,Amazon Elastic Compute Cloud EC2,Y,N
11867,Data Scientists,Amazon Redshift,Y,N
11868,Data Scientists,Amazon Simple Storage Service S3,N,N
11869,Data Scientists,Amazon Web Services AWS SageMaker,N,N
...,...,...,...,...
11947,Data Scientists,TensorFlow,Y,Y
11948,Data Scientists,Teradata Database,Y,N
11949,Data Scientists,The MathWorks MATLAB,Y,Y
11950,Data Scientists,UNIX,Y,N


This will show software skills associated with the Data Scientists occupation.

In [ ]:
Step 7: Save your cleaned datasets

In [9]:
from pathlib import Path

# Output folder
output_folder = Path(
    r"C:\Users\belie\OneDrive\Desktop\ADT MySQL\Lab 7 Project Milestone\JobAvenir\data\cleaned"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

# Save O*NET datasets
occupation_clean.to_csv(
    output_folder / "occupations_clean.csv",
    index=False
)

software_clean.to_csv(
    output_folder / "software_skills_clean.csv",
    index=False
)

job_titles_clean.to_csv(
    output_folder / "job_titles_clean.csv",
    index=False
)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!
